# Recorte de enunciados de PDFs de emestrada — paso a paso

In [1]:
# %pip install pdfplumber img2pdf pillow pandas

import io
import re
import os
import shutil
from pathlib import Path
import unicodedata

import pandas as pd
import pdfplumber
import img2pdf
from PIL import Image
from IPython.display import display

## Establecer directorio raiz

In [2]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

## Parámetros para medidas de recuadros y pdf

In [3]:
ANCHO_MIN, ALTO_MIN = 450, 20   # tamaño mínimo para considerar un rectángulo como recuadro de enunciado
MARGEN = 2                      # holgura alrededor del borde al recortar
DPI = 200                       # resolución de los PNG
A4 = (595.28, 841.89)
MARGEN_A4 = 40                  # margen de la hoja de salida
SEPARACION = 15                 # hueco entre recuadros apilados

## Función para detectar los recuadros

In [4]:
def detect_rectangles(page):
    vistos = {}
    for r in page.rects:
        if r["width"] > ANCHO_MIN and r["height"] > ALTO_MIN:
            clave = tuple(round(r[k]) for k in ("x0", "top", "x1", "bottom"))
            vistos[clave] = r
    return sorted(vistos.values(), key=lambda r: r["top"])

In [5]:
def norm(s: str) -> str:
    s = unicodedata.normalize("NFKD", s.upper())
    s = "".join(c for c in s if not unicodedata.combining(c))  # quita tildes
    return re.sub(r"\s+", "", s)

## Extraer y procesar los datos del examen

In [6]:
def process_exercise_info(exercise_info):
    exercise_info = exercise_info.lower()

    # valores por defecto si no puede procesarse
    # la información del ejercicio
    (asignatura, year, convocatoria, ejercicio) = ("" for _ in range(4))
    
    asignaturas = {
        "quimica" : "Química",
        "química" : "Química",
        "física" : "Física",
        "fisica" : "Física",
        "sociales ii" : "Mates CCSS",
        "matematicas ii" : "Matemáticas II"
    }

    # extraer asignatura
    for k,v in asignaturas.items():
        if k in exercise_info:
            asignatura = v
            break
        else:
            asignatura = "asignatura_desconocida"

    # extraer año
    year_pattern = r"2\d{3}"
    year = re.findall(year_pattern, exercise_info)[0]

    # extraer convocatoria y ejercicio
    convocatorias = ["junio", "julio", "septiembre",
                     "modelo", "ponencia",
                     "reserva"]

    # convocatorias de Junio, Julio, Septiembre, modelo o ponencia
    for conv in convocatorias[:-1]:
        if conv in exercise_info:
            convocatoria = conv.title()
            convocatoria_position = exercise_info.find(conv)
            ejercicio = (" ".join(exercise_info[convocatoria_position:].split(" ")[1:])
                         .title()
            )
            break

    # convocatoria es una reserva
    if "reserva" in exercise_info:
        convocatoria_position = exercise_info.find("reserva")
        convocatoria = (" ".join(exercise_info[convocatoria_position:]
                                 .split(" ")[:2])
                                 .title()
                                 .replace(".", "")
        )
        ejercicio = (" ".join(
            exercise_info[convocatoria_position:].split(" ")[2:]
            ).title()
        )
    
    return (asignatura, year, convocatoria, ejercicio)   


Limpieza de texto

In [7]:
def slug(texto):
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return re.sub(r"\W+", "_", texto).strip("_").lower()

#print(slug("RESERVA 1"), "|", slug("SEPTIEMBRE"), "|", slug("2023 - Integrales"))

## Composición del PDF de enunciados a partir de las imágenes (`img2pdf`).

Se rasteriza para que el PDF no arrastre el contenido oculto de la página original (soluciones).

In [8]:
def _a_png(img, dpi):
    buf = io.BytesIO()
    img.save(buf, format="PNG", dpi=(dpi, dpi))  # el dpi embebido fija el tamaño físico en img2pdf
    return buf.getvalue()


def componer_pdf(imagenes, salida, dpi):
    """Apila las imágenes en hojas A4, una debajo de otra, saltando de hoja cuando no caben."""
    px = dpi / 72                                   # puntos -> píxeles
    ancho, alto = round(A4[0] * px), round(A4[1] * px)
    margen, sep = round(MARGEN_A4 * px), round(SEPARACION * px)
    util_w, util_h = ancho - 2 * margen, alto - 2 * margen

    paginas, hoja, y = [], None, None
    for img in imagenes:
        escala = min(1.0, util_w / img.width, util_h / img.height)
        if escala < 1:
            img = img.resize((round(img.width * escala), round(img.height * escala)), Image.LANCZOS)
        if hoja is None or y + img.height > alto - margen:
            if hoja is not None:
                paginas.append(_a_png(hoja, dpi))
            hoja = Image.new("RGB", (ancho, alto), "white")
            y = margen
        hoja.paste(img, ((ancho - img.width) // 2, y))
        y += img.height + sep
    paginas.append(_a_png(hoja, dpi))

    salida.write_bytes(img2pdf.convert(paginas))

## Procesar archivo pdf

In [9]:
ruta = Path('/Users/daniel/code/clasificador_selectividad/source_pdf_files/Física - Temas/campo_eléctrico_magnético/2016 - Campo eléctrico y magnético.pdf')

In [10]:
parts = list(ruta.parts)
parts.reverse()
parts

['2016 - Campo eléctrico y magnético.pdf',
 'campo_eléctrico_magnético',
 'Física - Temas',
 'source_pdf_files',
 'clasificador_selectividad',
 'code',
 'daniel',
 'Users',
 '/']

In [11]:
def procesar_pdf(ruta_pdf, output_folder, export_pdf = True, export_png=True, dpi=DPI):
    exercises, imagenes = [], []

    # ruta_pdf
    #'/Users/daniel/code/clasificador_selectividad/source_pdf_files/Física - Temas/campo_eléctrico_magnético/2016 - Campo eléctrico y magnético.pdf'

    # output_folder
    # clasificador_selectividad/enunciados_ejercicios/Física - Temas/campo_gravitatorio

    path_parts = list(ruta_pdf.parts)
    path_parts.reverse()
    subject_path = path_parts[2].replace(" - Temas", "")
    topic_path = path_parts[1]
    png_folder = output_folder / "png" / f"png_{subject_path}_{ruta_pdf.stem}"

    # 1) Detección con pdfplumber
    with pdfplumber.open(ruta_pdf) as pdf:
        # procesamiento de las páginas del pdf
        for n, page in enumerate(pdf.pages):
            # detectar los recuadros
            boxes = detect_rectangles(page)

            # obtener coordenadas del recuadro
            for k, r in enumerate(boxes, start=1):
                x0 = max(r["x0"] - MARGEN, 0)
                x1 = min(r["x1"] + MARGEN, page.width)
                top = max(r["top"] - MARGEN, 0)
                bottom = min(r["bottom"] + MARGEN, page.height)

                # recortar el trozo del recuadro del pdf
                clip = page.crop((x0, top, x1, bottom))

                # extraer el texto del enunciado en
                # el recuadro
                texto_raw = clip.extract_text() or ""

                # separar el enunciado en líneas
                lineas = texto_raw.split("\n")

                # el enunciado son todas las líneas excepto
                # la última (información del ejercicio)
                texto = "\n".join(lineas[:-1])

                # información del ejercicio
                exercise_info = lineas[-1]
                asignatura, anio, conv, ej = process_exercise_info(exercise_info)
                # texto = texto_raw.replace("\n", " ")
                # metadata = PATRON.search(texto_raw)   # buscar con saltos de línea

                # crear nombre de archivo png
                # a partir de la información del ejercicio
                if exercise_info:
                    ej_slug = slug(ej)
                    png_name = f"{asignatura}_{topic_path}_{anio}_{slug(conv)}_{ej_slug}"
                    # ej_slug = slug(re.sub(r"EJERCICIO", "", metadata["ej"], flags=re.IGNORECASE))
                    # nombre = f"{metadata['anio']}_{slug(metadata['conv'])}_{ej_slug}"
                else:
                    png_name = f"{slug(ruta_pdf.stem)}_p{n + 1:02d}"

                # rasterizar el recorte (se usa para el png y para el pdf)
                imagen_pil = None
                if export_png or export_pdf:
                    imagen_pil = clip.to_image(resolution=dpi).original.convert("RGB")

                # exportar recorte a png
                image = None
                if export_png:
                    png_folder.mkdir(parents=True, exist_ok=True)
                    output_file = png_folder / f"{png_name}.png"
                    i = 2
                    while output_file.exists():
                        output_file = png_folder / f"{png_name}_{i}.png"
                        i += 1
                    imagen_pil.save(output_file, dpi=(dpi, dpi))
                    image = str(output_file.relative_to(output_folder))

                if export_pdf:
                    imagenes.append(imagen_pil)

                aviso = ""
                if not exercise_info:
                    aviso = "sin info ejercicio"
                elif len(boxes) > 1:
                    aviso = f"{len(boxes)} recuadros en la página"

                # asignatura y tema
                #asignatura = ruta_pdf.parts[-3].replace(" - Temas", "").replace(" ", "").lower()
                tema = ruta_pdf.stem.split(" - ")[1]
                if asignatura == "asignatura_desconocida":
                    asignatura = subject_path

                # exercise information dictionary
                exercise_info_dict = {
                    "asignatura": asignatura,
                    "asignatura_carpeta" : subject_path,
                    "año" : anio,
                    "convocatoria" : conv,
                    "ejercicio" : ej,
                    "tema": tema,
                    "info_ejercicio" : exercise_info,
                    "enunciado": texto,
                    "archivo": "/".join(ruta_pdf.parts[-3:]),
                    "imagen": image,
                    "aviso": aviso,
                    #"año": metadata["anio"] if metadata else "",
                    #"convocatoria": metadata["conv"].title() if metadata else "",
                    #"ejercicio": metadata["ej"] if metadata else "",
                }

                # eliminar la clave imagen si export_png = False
                if not export_png:
                    del exercise_info_dict["imagen"]
                    
                exercises.append(exercise_info_dict)

    if not exercises:
        return pd.DataFrame(), "ningún recuadro detectado"

    # combinar todos los enunciados en un pdf (raster, desde las imágenes)
    if export_pdf:
        carpeta_pdf = output_folder
        carpeta_pdf.mkdir(parents=True, exist_ok=True)
        output_pdf_filename = carpeta_pdf / f"{subject_path}_{ruta_pdf.stem}_enunciados.pdf"
        componer_pdf(imagenes, output_pdf_filename, dpi)

    return pd.DataFrame(exercises), ""

## Procesar carpeta

In [12]:
def process_folder(folder, export_pdf = True, export_png = True, files_to_process_limit=None):

    # '/home/daniel/code/clasificador_selectividad/source_pdf_files/Física - Temas/campo_gravitatorio'
    source_folder = Path(folder)
    output_folder_name = "/".join(source_folder.parts[-2:]).replace(" - Temas", "")
    output_folder = Path(f"{root}/enunciados_ejercicios/{output_folder_name}")
    output_folder.mkdir(parents=True, exist_ok=True)
    print(f"Procesando carpeta: {output_folder_name}")

    # obtener rutas de los archivos pdf de la carpeta
    pdf_files = sorted(p for p in source_folder.rglob("*.pdf") if not p.stem.endswith("_enunciados"))
    
    # hay límite de archivos por procesar
    if files_to_process_limit:
        pdf_files = pdf_files[:files_to_process_limit]

    print(f"{len(pdf_files)} PDFs a procesar\n")

    dataframes, issues = [], []
    for index, file_path in enumerate(pdf_files, start=1):
        try:
            df, aviso = procesar_pdf(file_path, output_folder, export_pdf = export_pdf, export_png = export_png)
        except Exception as e:
            subject = output_folder.parts[-2]
            issues.append({"issue": f"ERROR: {e}",
                           "subject" : subject,
                           "pdf file": f"{subject}_{file_path.name}"
                           })
            print(f"[{index}/{len(pdf_files)}] ERROR: {subject} - {file_path.name}: {e}")
            continue

        dataframes.append(df)

        if aviso:
            issues.append({"issue": aviso,
                           "pdf file": str(file_path)})
            
            print(f"[{index}/{len(pdf_files)}] AVISO  {file_path.name}: {aviso}")
        else:
            print(f"[{index}/{len(pdf_files)}] OK     {file_path.name}: {len(df)} ejercicios procesados")

    # Guardar resultados en CSV
    enunciados_df = pd.DataFrame()
    for d in dataframes:
        enunciados_df = pd.concat([enunciados_df, d])
            
    enunciados_df.to_csv(output_folder / f"{output_folder.name}.csv", index=False)

    if issues:
        issues_df = pd.DataFrame(issues)
        issues_df.to_csv(output_folder / f"{output_folder.name}_issues.csv",
                            index=False, encoding="utf-8")

    print("*" * 40)
    print(f"\nProcesamiento completado. Resultados guardados en: {output_folder.relative_to(root / "enunciados_ejercicios")}\n\n")

In [13]:
# process_folder(root/"source_pdf_files/Mates CCSS - Temas/sistemas_ecuaciones_lineales_ccss",
#                export_pdf=True)

## Procesar carpetas

Obtener las rutas de los archivos

In [14]:
os.chdir(f"{root}/source_pdf_files")

In [15]:
carpetas_asignaturas = {d.name:[] for d in Path(".").iterdir() if d.is_dir()}

In [16]:
for asignatura in carpetas_asignaturas:
    carpetas_asignaturas[asignatura] = [d.name for d in Path(f"./{asignatura}").iterdir() if d.is_dir()]

In [17]:
folder_paths = []
for asignatura, carpetas in carpetas_asignaturas.items():
    for carpeta in carpetas:
        folder_paths.append(f"{root}/source_pdf_files/{asignatura}/{carpeta}")

In [18]:
folder_paths[:2]

['/Users/daniel/code/clasificador_selectividad/source_pdf_files/Física - Temas/campo_eléctrico_magnético',
 '/Users/daniel/code/clasificador_selectividad/source_pdf_files/Física - Temas/física_cuántica_nuclear']

## Procesar todos los archivos pdf

In [19]:
# process_folder(folder_paths[0], files_to_process_limit=None)

In [20]:
folders_count = len(folder_paths)
for index, folder in enumerate(folder_paths, start=1):
    # Limitar a 5 archivos por carpeta para pruebas
    process_folder(folder, export_pdf = True, export_png = True, files_to_process_limit=None)
    remaining_folders = folders_count - index
    if remaining_folders == 0:
        print("*" * 5 + "Todos los archivos han sido procesados." + "*" * 5)
    else:
        print(f"\n\nQuedan {remaining_folders} carpetas por procesar.")

Procesando carpeta: Física/campo_eléctrico_magnético
10 PDFs a procesar

[1/10] OK     2016 - Campo eléctrico y magnético.pdf: 10 ejercicios procesados
[2/10] OK     2017 - Campo eléctrico y magnético.pdf: 12 ejercicios procesados
[3/10] OK     2018 - Campo eléctrico y magnético.pdf: 12 ejercicios procesados
[4/10] OK     2019 - Campo eléctrico y magnético.pdf: 12 ejercicios procesados
[5/10] OK     2020 - Campo eléctrico y magnético.pdf: 12 ejercicios procesados
[6/10] OK     2021 - Campo eléctrico y magnético.pdf: 12 ejercicios procesados
[7/10] OK     2022 - Campo eléctrico y magnético.pdf: 12 ejercicios procesados
[8/10] OK     2023 - Campo eléctrico y magnético.pdf: 12 ejercicios procesados
[9/10] OK     2024 - Campo eléctrico y magnético.pdf: 12 ejercicios procesados
[10/10] OK     2025 - Campo eléctrico y magnético.pdf: 15 ejercicios procesados
****************************************

Procesamiento completado. Resultados guardados en: Física/campo_eléct

## Mover archivos .csv

In [21]:
Path(root / "csv_enunciados").mkdir(exist_ok=True)
os.chdir(root / "enunciados_ejercicios")

In [22]:
for file in Path(".").rglob("*.csv"):
    subject = file.parts[-3].replace(" - Temas", "").lower()    
    new_filename = f"{subject}_{file.name}"
    output_csv_path = root / f"csv_enunciados/{new_filename}"
    shutil.move(file, output_csv_path)

## Combinar todos los archivos csv

In [23]:
issues_df = pd.DataFrame()
exercises_df = pd.DataFrame()

# moverse al directorio donde se encuentran los CSV
os.chdir(root / "csv_enunciados")

for file in Path(".").rglob("*.csv"):
    if "issue" in file.name:
        df = pd.read_csv(file, encoding = "utf-8")
        issues_df = pd.concat([issues_df, df])
    else:
        df = pd.read_csv(file, encoding = "utf-8")
        exercises_df = pd.concat([exercises_df, df])

# reordenar filas de exercises_df por asignatura, año, convocatoria y ejercicio
exercises_df = exercises_df.sort_values(by=["asignatura", "año", "convocatoria", "ejercicio"]).reset_index(drop=True)

if not issues_df.empty:
    issues_df.to_csv(root / "csv_enunciados/issues.csv", index=False, encoding="utf-8")

if not exercises_df.empty:
    exercises_df.to_csv(root / "csv_enunciados/ejercicios_enunciados.csv", index=False, encoding="utf-8")

# comprobar existencia de issues.csv y si tiene contenido
# borrar los archivos _issues.csv
if (root / "csv_enunciados/issues.csv").exists() and os.path.getsize(root / "csv_enunciados/issues.csv") > 0:
    issue_files = Path(".").rglob("*_issues.csv")
    # clean issue files
    for f in issue_files:
        f.unlink()

# comprobar existencia de ejercicios_enunciados.csv y si tiene contenido
# borrar los archivos *.csv
if (root / "csv_enunciados/ejercicios_enunciados.csv").exists() and os.path.getsize(root / "csv_enunciados/ejercicios_enunciados.csv") > 0:
    csv_files = [f for f in Path(".").rglob("*.csv") if f.name not in ["ejercicios_enunciados.csv", "issues.csv"] ]
    # clean csv files
    for f in csv_files:
        f.unlink()

In [24]:
issues_df

,issue,subject,pdf file
0,ERROR: list index out of range,Mates CCSS,Mates CCSS_2016 - Funciones CCSS.pdf
0,ERROR: list index out of range,Química,Química_2021 - Enlace Químico.pdf
0,ERROR: list index out of range,Química,Química_2008 - Reactividad Orgánica.pdf
0,ningún recuadro detectado,NaN,/Users/daniel/code/clasificador_selectividad/s...


In [25]:
df = pd.read_csv(root / "csv_enunciados/ejercicios_enunciados.csv")
df.sample(5)

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
2404,Mates II,Mates II,2014,Reserva 4,Ejercicio 4. Opción B.,Geometría,MATEMÁTICAS II. 2014. RESERVA 4. EJERCICIO 4. ...,x5 z6\nConsidera el plano  de ecuación 2x ...,Mates II - Temas/geometría/2014 - Geometría.pdf,png/png_Mates II_2014 - Geometría/asignatura_...,NaN
3349,Química,Química,2005,Reserva 1,Ejercicio 6. Opción A,Ácido Base,QUIMICA. 2005. RESERVA 1. EJERCICIO 6. OPCIÓN A,Al disolver 0’23 g de HCOOH en 50 mL de agua s...,Química - Temas/ácido_base/2005 - Ácido Bas...,png/png_Química_2005 - Ácido Base/Química_á...,NaN
3024,Química,Química,2000,Reserva 4,Ejercicio 6. Opción A,Equilibrio Químico,QUÍMICA. 2000. RESERVA 4. EJERCICIO 6. OPCIÓN A,"A 523 ºK las concentraciones de PCl ,PCl y Cl ...",Química - Temas/equilibrio_químico/2000 - Eq...,png/png_Química_2000 - Equilibrio Químico/Qu...,NaN
3376,Química,Química,2005,Reserva 4,Ejercicio 1 Opción B,Formulación,QUIMICA. 2005. RESERVA 4 EJERCICIO 1 OPCIÓN B,Formule o nombre los compuestos siguientes: a)...,Química - Temas/formulación/2005 - Formulaci...,png/png_Química_2005 - Formulación/Química_f...,NaN
183,Física,Física,2019,Septiembre,Ejercicio 4. Opción B,Física cuántica y nuclear,FISICA. 2019. SEPTIEMBRE. EJERCICIO 4. OPCIÓN B,"g = 9,8 m s-2\na) Explique el proceso de conse...",Física - Temas/física_cuántica_nuclear/2019...,png/png_Física_2019 - Física cuántica y nuc...,NaN


In [26]:
df.asignatura.value_counts()

asignatura
Química       1808
Mates II      1267
Mates CCSS    1213
Física         486
Name: count, dtype: int64

In [27]:
os.chdir(root / "enunciados_ejercicios")
pdf_sizes = [i.stat().st_size for i in Path(".").rglob("*.pdf")]
import numpy as np
print(f"Total PDF files: {len(pdf_sizes)}")
print(f"Average size: {round(np.mean(pdf_sizes) / 1024**2, 2)} MB")

Total PDF files: 574
Average size: 0.54 MB
